To view your notebook as a Voilà web application, replace the word "notebooks" in your browser's URL with: "voila/render". You will see the same content as your notebook, but without any of the code cells.

In [ ]:
# #hide
# ! [ -e /content ] && pip install -Uqq fastbook
# import fastbook
# fastbook.setup_book()

In [1]:
#hide
# from fastbook import *
from fastai.vision.all import *
from fastai.vision.widgets import *

In [2]:
# The plum compatibility patch for python 3.12 with 3.14
# Your model was saved on Python 3.12 with an older plum where Resolver state was restored via __dict__.
import plum._resolver

class _ResolverCompat(plum._resolver.Resolver):
    def __setstate__(self, state):
        if isinstance(state, dict):
            for key, value in state.items():
                setattr(self, key, value)

plum._resolver.Resolver = _ResolverCompat

In [3]:
path = Path()
learn_inf = load_learner(path/'models/01-cat-or-dog-model.pkl')

/opt/miniconda3/envs/AI_models/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [ ]:
# #hide_output
# btn_upload = widgets.FileUpload()

In [ ]:
# img = PILImage.create(btn_upload.data[-1])

In [ ]:
# pred,pred_idx,probs = learn_inf.predict(img)

In [ ]:
# #hide_output
# lbl_pred = widgets.Label()
# lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

In [ ]:
# #hide_output
# btn_run = widgets.Button(description='Classify')

In [ ]:
# def on_click_classify(change):
#     img = PILImage.create(btn_upload.data[-1])
#     out_pl.clear_output()
#     with out_pl: display(img.to_thumb(128,128))
#     pred,pred_idx,probs = learn_inf.predict(img)
#     lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

# btn_run.on_click(on_click_classify)

In [ ]:
# #hide
# #Putting back btn_upload to a widget for next cell
# btn_upload = widgets.FileUpload()

In [ ]:
# #hide_output
# VBox([widgets.Label('Select your bear!'), 
#     btn_upload, btn_run, out_pl, lbl_pred])

In [4]:
from io import BytesIO

import ipywidgets as widgets
from IPython.display import clear_output, display

_CLASS_NAMES = {False: 'Dog', True: 'Cat'}

learn_inf.dls.cpu()
_ = learn_inf.model.cpu()


def safe_predict(learn, img):
    learn.model.eval()
    x = learn.dls.after_item(img)
    xb = learn.dls.after_batch(torch.stack([x]))
    with torch.inference_mode():
        probs = learn.model(xb).softmax(dim=-1)[0]
    pred_idx = int(probs.argmax())
    return learn.dls.vocab[pred_idx], pred_idx, probs


def thumb_png_bytes(img) -> bytes:
    thumb = img.to_thumb(128, 128)
    if thumb.mode not in ('RGB', 'L'):
        thumb = thumb.convert('RGB')
    buf = BytesIO()
    thumb.save(buf, format='PNG')
    return buf.getvalue()


btn_upload = widgets.FileUpload(accept='image/*')
btn_run = widgets.Button(description='Classify')
img_hint = widgets.Label(value='Thumbnail appears here after Classify.')
img_out = widgets.Image(
    format='png',
    width=128,
    height=128,
    layout=widgets.Layout(display='none'),
)
lbl_pred = widgets.Label(value='Upload an image, then click Classify.')


def on_click_classify(_change):
    if not btn_upload.value:
        lbl_pred.value = 'Please upload an image first.'
        return

    try:
        content = btn_upload.value[0]['content']
        if isinstance(content, memoryview):
            content = bytes(content)

        img = PILImage.create(BytesIO(content))
        pred, pred_idx, probs = safe_predict(learn_inf, img)

        img_out.value = thumb_png_bytes(img)
        img_hint.layout.display = 'none'
        img_out.layout.display = ''

        label = _CLASS_NAMES.get(pred, str(pred))
        prob = float(probs[pred_idx])
        lbl_pred.value = f'Prediction: {label}; Probability: {prob:.04f}'
    except Exception as exc:
        lbl_pred.value = f'Prediction failed: {exc}'


btn_run.on_click(on_click_classify)

# Clear this cell's output so re-running does not leave multiple VBox copies on screen.
clear_output(wait=True)
display(widgets.VBox([
    widgets.Label('Select your picture!'),
    btn_upload,
    btn_run,
    img_hint,
    img_out,
    lbl_pred,
]))

In [ ]:

# !pip install voila
# !jupyter server extension enable --sys-prefix voila